# 10 임베딩 기반 시계열 분석 (Phase 2)

## Phase2 base = LSTM + 임베딩

Phase1(07–09)에서 **ARIMA가 조건별 Best 1위**(mean MAPE 기준 LSTM과 사실상 공동 선두)지만, **ARIMA·SBA·TSB·Prophet은 단변량**이라 시계열 임베딩(제품별 고정 벡터)을 **유효하게 결합할 수 없습니다**(정적 exog는 상수라 절편만 이동).

임베딩을 결합할 수 있는 모델(패널 ML·신경망) 중, **LSTM이 Phase1 성능 최상위(mean MAPE 최저)** 이면서 **임베딩(static exog)으로 성능을 더 끌어올릴 여지**가 큽니다. 따라서 **실질적 best-1 = LSTM**을 Phase2 대표 base로 고정하고 6종 임베딩을 결합해 비교합니다.

> 논문(§4.4)은 Phase2 base로 **iTransformer + 임베딩**을 사용했습니다. 본 실습의 **LSTM + 임베딩(static exog)** 은 같은 "신경망 + 시계열 임베딩" 계열로 논문 방식에 근접합니다.

| 축 | 내용 |
|----|------|
| type | **E(고변동), C(저변동)** |
| cluster | SBC(4) + ML(AE+KMeans, 2) |
| **조건** | **12** (SBC 8 + ML 4) |
| base (고정) | **LSTM** (neuralforecast, 임베딩 = static exog) |
| 임베딩 | PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST |
| 조합 | **12 × 6 = 72** |

**오차지표:** MAE, RMSE, MAPE, MASE — 조건별 Best 임베딩은 MAPE 최소 기준

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 66
학습 <= 201707 | 검증: [201708, 201709, 201710, 201711, 201712, 201713, 201714, 201715, 201716, 201717, 201718, 201719, 201720]
Best 선정 기준: MAPE


### Phase1 능력치 분석 — 왜 LSTM을 Phase2 base로 고정하는가

07~09장 Phase1(10모델 × 12조건)을 종합하면:

- **Phase1 최상위는 ARIMA·LSTM.** ARIMA는 조건별 통합 Best를 12조건 중 5개로 1위, **LSTM은 mean MAPE 45.1로 10모델 중 최저**(가장 robust — 간헐 family에서 오차 폭주가 적음). 둘은 사실상 공동 선두.
- **그러나 ARIMA·SBA·TSB·Prophet은 단변량**이라 시계열 임베딩(제품별 고정 벡터)을 결합해도 상수 exog로 절편만 이동 → **임베딩 하이브리드에 부적합**.
- **임베딩을 결합할 수 있는 모델 = 패널 ML(XGBoost·RF) + 신경망(LSTM·N-HiTS·Autoformer·iTransformer).** 이 중 **LSTM이 mean MAPE 최저**로 성능·robust 모두 우수.

→ 따라서 "**단변량이라 임베딩으로 개선 불가한 ARIMA**"가 아니라, "**임베딩으로 더 끌어올릴 수 있는 실질 best-1 = LSTM**"을 Phase2 대표 base로 고정합니다. LSTM은 neuralforecast의 **static exog**로 임베딩을 결합해 학습합니다(논문 iTransformer+임베딩 계열).

In [2]:
# === Phase1 10모델 능력치 분석 (Phase2 base 선정 근거) ===
phase1_all = pd.read_parquet(DATA_PROCESSED / 'phase1_all_results.parquet')
phase1_best = pd.read_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv')

# 임베딩을 결합 가능한 모델 = 단변량 통계(ARIMA/SBA/TSB/Prophet)를 제외한 나머지
UNIVARIATE = ['ARIMA', 'SBA', 'TSB', 'Prophet']

perf = (phase1_all.groupby('model')['mape']
        .agg(median_mape='median', mean_mape='mean').round(1))
perf['embedding_capable'] = ~perf.index.isin(UNIVARIATE)
print('=== [1] Phase1 10모델 성능 (mean MAPE 오름차순, mean은 robust 지표) ===')
display(perf.sort_values('mean_mape'))

print('=== [2] 조건별 통합 Best 모델 빈도 (12조건) ===')
display(phase1_best['best_model'].value_counts().rename('n_conditions').to_frame())

print('=== [3] 임베딩 결합 가능 모델만 (mean MAPE 순) → Phase2 base 후보 ===')
display(perf[perf['embedding_capable']].sort_values('mean_mape')[['median_mape', 'mean_mape']])

print('[결론] ARIMA는 Phase1 상위이나 단변량 -> 임베딩 결합 불가.')
print('       임베딩 결합 가능 모델 중 LSTM이 mean MAPE 최저(robust) -> Phase2 base = LSTM.')

=== [1] Phase1 10모델 성능 (mean MAPE 오름차순, mean은 robust 지표) ===


,median_mape,mean_mape,embedding_capable
model,,,
LSTM,34.0,45.1,True
ARIMA,32.9,45.7,False
XGBoost,34.6,99.1,True
N-HiTS,32.6,99.4,True
Prophet,36.5,113.0,False
RF,36.9,131.0,True
Autoformer,39.7,137.3,True
iTransformer,33.8,142.0,True
SBA,31.4,161.9,False


=== [2] 조건별 통합 Best 모델 빈도 (12조건) ===


,n_conditions
best_model,
ARIMA,5
Prophet,2
LSTM,2
RF,2
N-HiTS,1


=== [3] 임베딩 결합 가능 모델만 (mean MAPE 순) → Phase2 base 후보 ===


,median_mape,mean_mape
model,,
LSTM,34.0,45.1
XGBoost,34.6,99.1
N-HiTS,32.6,99.4
RF,36.9,131.0
Autoformer,39.7,137.3
iTransformer,33.8,142.0


[결론] ARIMA는 Phase1 상위이나 단변량 -> 임베딩 결합 불가.
       임베딩 결합 가능 모델 중 LSTM이 mean MAPE 최저(robust) -> Phase2 base = LSTM.


### ① LSTM × 6임베딩 — 12조건 실험

In [3]:
from utils.phase_experiments import REPRESENTATIVE_BASE_MODEL

BASE = REPRESENTATIVE_BASE_MODEL
print('대표 base:', BASE)

p2_cache = DATA_PROCESSED / 'phase2_results.parquet'
if p2_cache.exists():
    phase2 = pd.read_parquet(p2_cache)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    print('Phase2 캐시 로드 |', len(phase2), 'rows')
else:
    emb_cache = build_global_embedding_cache(df)
    p2_sbc = run_phase2_all(df, feat_df, 'SBC_CLUSTER', 'SBC', fixed_model=BASE, emb_cache=emb_cache)
    p2_ml = run_phase2_all(df, feat_df, 'ML_CLUSTER', 'ML', fixed_model=BASE, emb_cache=emb_cache)
    phase2 = pd.concat([p2_sbc, p2_ml], ignore_index=True)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    phase2.to_parquet(p2_cache, index=False)
    phase2_summary.to_csv(DATA_PROCESSED / 'phase2_summary.csv', index=False)
    phase2_best.to_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv', index=False)
    print('Phase2 완료 |', len(phase2), 'rows')

print('유효 조건(제품 있음):', phase2.groupby(['cluster_scheme','type','cluster']).ngroups)
display(phase2_best.sort_values(['cluster_scheme', 'type', 'cluster']))


대표 base: LSTM


Global embeddings:   0%|          | 0/6 [00:00<?, ?it/s]

Phase2 SBC:   0%|          | 0/48 [00:00<?, ?it/s]

Seed set to 42


GPU available: True (cuda), used: True


...

TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Phase2 완료 | 792 rows
유효 조건(제품 있음): 12


C:\Users\kjh\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


,cluster_scheme,type,cluster,best_hybrid,base_model,embedding,mae_mean,rmse_mean,best_mape,mase_mean
1,ML,C,1,LSTM+FastDTW,LSTM,FastDTW,32101.474356,49836.435661,28.876633,2.282851
6,ML,C,2,LSTM+AE,LSTM,AE,996.525849,1865.355211,39.180868,1.790653
16,ML,E,1,LSTM+PatchTST,LSTM,PatchTST,13675.558010,25681.270857,24.055970,1.496677
19,ML,E,2,LSTM+FastDTW,LSTM,FastDTW,1059.388316,1799.741627,47.628334,1.974296
26,SBC,C,1,LSTM+GAF-CNN,LSTM,GAF-CNN,6273.094296,9936.262737,39.676231,2.190193
31,SBC,C,2,LSTM+FastDTW,LSTM,FastDTW,929.892942,1869.445603,36.197626,1.680839
37,SBC,C,3,LSTM+FastDTW,LSTM,FastDTW,726.903892,1365.584345,34.832441,0.458754
44,SBC,C,4,LSTM+GAF-CNN,LSTM,GAF-CNN,18.744761,42.092954,44.138258,0.674556
48,SBC,E,1,LSTM+AE,LSTM,AE,2221.913774,3847.689722,44.483375,1.805872
58,SBC,E,2,LSTM+PatchTST,LSTM,PatchTST,570.246579,1070.460269,50.972340,2.286073


### ② 결과 요약

### ③ 11장 하이브리드 비교로의 연결

- 산출물 `phase2_best_per_condition.csv` = 조건별 **LSTM + Best 임베딩** (MAPE)
- 11장에서 SBC·ML scheme 각각을 type별 **제품수 가중 WMAPE**(논문 §4.5)로 합산 비교
- 동일 base·임베딩 파이프라인에서 clustering scheme만 다르므로 SBC vs ML 비교가 공정

In [4]:
print('=== 조건별 Best 임베딩 (MAPE) ===')
print(phase2_best['embedding'].value_counts())

print('\n=== 임베딩별 평균 4지표 (전 조건) ===')
emb_avg = phase2.groupby('embedding')[['mae','rmse','mape','mase']].mean().round(2)
display(emb_avg.sort_values('mape'))

print('\n=== scheme별 LSTM+임베딩 평균 MAPE ===')
print(phase2.groupby(['cluster_scheme','embedding'])['mape'].mean().unstack('cluster_scheme').round(2))


=== 조건별 Best 임베딩 (MAPE) ===
embedding
FastDTW     5
AE          3
PatchTST    2
GAF-CNN     2
Name: count, dtype: int64

=== 임베딩별 평균 4지표 (전 조건) ===


,mae,rmse,mape,mase
embedding,,,,
FastDTW,2679.28,4405.49,43.35,1.95
AE,2677.03,4437.16,43.82,1.91
GAF-CNN,2685.93,4425.56,44.03,1.92
PCA,2685.93,4425.56,44.03,1.92
TS2Vec,2685.93,4425.56,44.03,1.92
PatchTST,2557.71,4336.43,45.12,1.93



=== scheme별 LSTM+임베딩 평균 MAPE ===
cluster_scheme     ML    SBC
embedding                   
AE              44.51  43.13
FastDTW         43.05  43.65
GAF-CNN         44.59  43.46
PCA             44.59  43.46
PatchTST        45.63  44.61
TS2Vec          44.59  43.46
